# T4: 燃料補正後レースペース比較

## 概要
R01–R03のクリーンロングランラップタイムに燃料補正を適用し、真のレースペースを比較する。

### 燃料補正モデル（CLAUDE.md準拠・概算）

| パラメータ | 値 | 備考 |
|---|---|---|
| 燃料消費量 | 1.75 kg/ラップ | 1.5–2.0 の中央値 |
| タイム影響 | 0.035 秒/kg | 0.03–0.04 の中央値 |
| 1ラップあたりの効果 | **+0.06 秒/ラップ** | 燃料軽量化により速くなる量 |

```
FuelCorrectedTime = LapTime_sec + LapNumber × 0.06
```

ラップが進むほど燃料が軽くなり自然にタイムが改善するため、  
その分を**足し戻す**ことで「レーススタート時の燃料量での換算タイム」に統一する。

> ⚠️ **注意**: 燃料補正値は概算。チームごとの実際の燃料搭載量は非公開のため未考慮。  
> 分析結果は常に「燃料補正前/後」を明記して解釈すること。

---

### 分析内容
1. 燃料補正の適用と効果量の確認
2. ドライバー別・GP別の生ペース vs 補正後ペースの比較
3. 燃料補正前後のチームランキング変動
4. 補正後ペース散布図（ドライバー別・GP別）
5. 補正後ペース vs レース最終順位の相関分析
6. まとめグラフの出力

In [ ]:
# ライブラリインポート
import matplotlib
matplotlib.use('Agg')  # GUIなし環境（Jupyter以外での実行時に必要）

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# Jupyter環境ではインラインで表示する
%matplotlib inline

print(f'pandas: {pd.__version__}')
print(f'numpy:  {np.__version__}')

In [ ]:
# ============================================================
# パス・定数設定
# ============================================================
BASE_DIR = '/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised'
INPUT_CSV = os.path.join(BASE_DIR, 'notebooks/output/clean_longruns.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'notebooks/output')

# レース結果CSV（GP別）
RACE_RESULTS = {
    'R01_Australia': os.path.join(BASE_DIR, 'data/2026_R01_Australia/export/race_results.csv'),
    'R02_China':     os.path.join(BASE_DIR, 'data/2026_R02_China/export/race_results.csv'),
    'R03_Japan':     os.path.join(BASE_DIR, 'data/2026_R03_Japan/export/race_results.csv'),
}

# 燃料補正定数（概算、CLAUDE.md準拠）
FUEL_CORRECTION_PER_LAP = 0.06  # 秒/ラップ

# GP表示名
GP_LABELS = {
    'R01_Australia': 'R01 オーストラリア',
    'R02_China':     'R02 中国',
    'R03_Japan':     'R03 日本（鈴鹿）',
}

# チームカラー（F1 2026）
TEAM_COLORS = {
    'McLaren':      '#F47600',
    'Ferrari':      '#E8002D',
    'Red Bull':     '#3671C6',
    'Mercedes':     '#00D7B6',
    'Aston Martin': '#229971',
    'Williams':     '#0093CC',
    'Racing Bulls': '#6692FF',
    'Alpine':       '#0090FF',
    'Haas':         '#B6BABD',
    'Kick Sauber':  '#52E252',
}

# グラフスタイル（CLAUDE.md準拠）
BG   = '#1a1a2e'
CARD = '#1c1c25'
TEXT = '#ffffff'
DIM  = '#aaaaaa'
GRID = '#333355'
RED  = '#e10600'

# 日本語フォント設定（Hiragino Sans が使用可能）
import matplotlib.font_manager as fm
for font in fm.fontManager.ttflist:
    if 'Hiragino' in font.name:
        plt.rcParams['font.family'] = font.name
        print(f'日本語フォント設定: {font.name}')
        break

print('設定完了')

## 1. データ読み込み

In [ ]:
# クリーンロングランCSVを読み込む
laps_df = pd.read_csv(INPUT_CSV)
print(f'クリーンロングラン: {len(laps_df):,} ラップ')
print(f'GP: {sorted(laps_df["GP"].unique())}')
print(f'ドライバー数: {laps_df["Driver"].nunique()}')
laps_df.head(3)

In [ ]:
# レース結果CSVを読み込む（全GP）
result_dfs = []
for gp_key, path in RACE_RESULTS.items():
    df = pd.read_csv(path)
    df['GP'] = gp_key
    result_dfs.append(df)
    print(f'  {gp_key}: {len(df)} ドライバー')

results_df = pd.concat(result_dfs, ignore_index=True)
print(f'\n合計: {len(results_df)} 行')
results_df.head(3)

## 2. 燃料補正の適用

```
FuelCorrectedTime = LapTime_sec + LapNumber × 0.06
```

ラップ番号が大きいほど燃料が軽くなっているため、その分（自然なタイム改善分）を足し戻す。

In [ ]:
# 燃料補正を適用（補正前の値も保持する）
laps_df['FuelCorrectedTime'] = laps_df['LapTime_sec'] + laps_df['LapNumber'] * FUEL_CORRECTION_PER_LAP

# 効果量の確認
diff = laps_df['FuelCorrectedTime'] - laps_df['LapTime_sec']
print('燃料補正量（秒）の統計:')
print(f'  最小: {diff.min():.2f}秒 （最も早いラップ = ラップ1付近）')
print(f'  最大: {diff.max():.2f}秒 （最も遅いラップ）')
print(f'  平均: {diff.mean():.2f}秒')
print(f'  中央値: {diff.median():.2f}秒')
print()
print(f'補正定数: {FUEL_CORRECTION_PER_LAP} 秒/ラップ（概算）')
print('注意: 燃料搭載量はチームごとに異なる（非公開）。この補正は均一仮定。')

In [ ]:
# 補正量の可視化（LapNumber vs 補正量）
fig, ax = plt.subplots(figsize=(10, 4), facecolor=BG)
ax.set_facecolor(CARD)

sample = laps_df.sample(min(500, len(laps_df)), random_state=42)
for i, gp in enumerate(sorted(sample['GP'].unique())):
    gp_data = sample[sample['GP'] == gp]
    color = [RED, '#6688cc', '#66cc88'][i]
    ax.scatter(gp_data['LapNumber'], 
               gp_data['FuelCorrectedTime'] - gp_data['LapTime_sec'],
               color=color, alpha=0.5, s=20, label=GP_LABELS.get(gp, gp))

# 理論ライン
x_theory = np.arange(1, 60)
ax.plot(x_theory, x_theory * FUEL_CORRECTION_PER_LAP, 
        color='white', linestyle='--', linewidth=1.5, alpha=0.7, label='理論値 (0.06×ラップ番号)')

ax.set_xlabel('ラップ番号', color=TEXT)
ax.set_ylabel('燃料補正量（秒）', color=TEXT)
ax.set_title('燃料補正量 = ラップ番号 × 0.06秒（概算）', color=TEXT, fontsize=13)
ax.tick_params(colors=TEXT)
ax.grid(True, color=GRID, alpha=0.5)
for spine in ax.spines.values(): spine.set_edgecolor(GRID)
ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=TEXT, fontsize=9)
plt.tight_layout()
plt.show()

## 3. ドライバー別サマリー計算

各ドライバー・GPごとに:
- `RawMedianPace`: クリーンラップの生タイム中央値（燃料補正前）
- `FuelCorrectedMedianPace`: クリーンラップの燃料補正後タイム中央値
- `CleanLaps`: 使用ラップ数
- `RacePosition`: レース最終順位

In [ ]:
# ドライバー別サマリーを計算
rows = []
for (gp, driver), grp in laps_df.groupby(['GP', 'Driver']):
    team = grp['Team'].iloc[0]
    raw_median  = grp['LapTime_sec'].median()
    fuel_median = grp['FuelCorrectedTime'].median()
    clean_laps  = len(grp)

    # レース最終順位
    mask = (results_df['GP'] == gp) & (results_df['Abbreviation'] == driver)
    pos_series = results_df.loc[mask, 'Position']
    race_position = int(pos_series.iloc[0]) if len(pos_series) > 0 else None

    rows.append({
        'GP': gp, 'Driver': driver, 'Team': team,
        'RawMedianPace':         round(raw_median, 4),
        'FuelCorrectedMedianPace': round(fuel_median, 4),
        'CleanLaps':             clean_laps,
        'RacePosition':          race_position,
    })

summary_df = pd.DataFrame(rows)
print(f'サマリー行数: {len(summary_df)}')
summary_df.head(10)

In [ ]:
# 補正前後のペース差を追加
summary_df['PaceDiff'] = summary_df['FuelCorrectedMedianPace'] - summary_df['RawMedianPace']

print('燃料補正後ペース - 補正前ペース（秒）の統計:')
print(summary_df.groupby('GP')['PaceDiff'].describe().round(3))

## 4. 燃料補正前後のチームランキング変動

In [ ]:
# チーム別ランキングを計算（各GP内でのベストドライバー基準）
team_rows = []
for (gp, team), grp in summary_df.groupby(['GP', 'Team']):
    raw_best  = grp['RawMedianPace'].min()
    fuel_best = grp['FuelCorrectedMedianPace'].min()
    team_rows.append({'GP': gp, 'Team': team,
                      'RawBestPace': raw_best, 'FuelBestPace': fuel_best})

team_df = pd.DataFrame(team_rows)
team_df['RawRank']  = team_df.groupby('GP')['RawBestPace'].rank(method='min')
team_df['FuelRank'] = team_df.groupby('GP')['FuelBestPace'].rank(method='min')
team_df['RankChange'] = team_df['RawRank'] - team_df['FuelRank']  # 正=補正後に順位上昇

# 全GP平均のランキング変動
team_avg = team_df.groupby('Team')[['RawRank', 'FuelRank', 'RankChange']].mean().sort_values('FuelRank')
team_avg.columns = ['補正前ランク（平均）', '補正後ランク（平均）', 'ランク変動（+↑）']
team_avg.round(2)

In [ ]:
# GP別ランキング詳細
for gp in sorted(team_df['GP'].unique()):
    print(f'\n=== {GP_LABELS.get(gp, gp)} ===')
    gp_team = team_df[team_df['GP'] == gp].sort_values('FuelRank')
    for _, row in gp_team.iterrows():
        arrow = '↑' if row['RankChange'] > 0 else ('↓' if row['RankChange'] < 0 else '→')
        print(f"  {int(row['FuelRank']):2d}位 {row['Team']:<20} "
              f"補正前={row['RawBestPace']:.3f}秒  補正後={row['FuelBestPace']:.3f}秒  "
              f"{arrow}{abs(row['RankChange']):.0f}")

## 5. 補正後ペース vs レース最終順位の相関分析

In [ ]:
# 相関係数（Spearman）: 補正前 vs 補正後
valid = summary_df.dropna(subset=['RacePosition'])

print('相関分析: ペース中央値 vs レース最終順位')
print()
for gp in sorted(valid['GP'].unique()):
    gp_data = valid[valid['GP'] == gp]
    rho_raw,  p_raw  = stats.spearmanr(gp_data['RawMedianPace'],          gp_data['RacePosition'])
    rho_fuel, p_fuel = stats.spearmanr(gp_data['FuelCorrectedMedianPace'], gp_data['RacePosition'])
    print(f'{GP_LABELS.get(gp, gp)}:')
    print(f'  補正前: ρ={rho_raw:.3f}  (p={p_raw:.4f})')
    print(f'  補正後: ρ={rho_fuel:.3f} (p={p_fuel:.4f})')
    print()

# 全GP合算
rho_raw_all,  p_raw_all  = stats.spearmanr(valid['RawMedianPace'],          valid['RacePosition'])
rho_fuel_all, p_fuel_all = stats.spearmanr(valid['FuelCorrectedMedianPace'], valid['RacePosition'])
print('=== 全GP合算 ===')
print(f'  補正前: ρ={rho_raw_all:.3f}  (p={p_raw_all:.4f})')
print(f'  補正後: ρ={rho_fuel_all:.3f} (p={p_fuel_all:.4f})')

## 6. 総合グラフの生成・保存

In [ ]:
# ====================================================
# 総合グラフ: 4パネル構成
# ====================================================
fig = plt.figure(figsize=(16, 12), facecolor=BG)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
axes = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(2)]

# 共通スタイル適用
for ax in axes:
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TEXT, labelsize=10)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    for spine in ax.spines.values(): spine.set_edgecolor(GRID)
    ax.grid(True, color=GRID, linewidth=0.5, alpha=0.7)

gp_list = sorted(summary_df['GP'].unique())
gp_colors = [RED, '#6688cc', '#66cc88']

# ---- [0] ドライバー別燃料補正後ペース散布図（GP別） ----
ax0 = axes[0]
marker_styles = ['o', 's', '^']

for i, gp in enumerate(gp_list):
    gp_data = summary_df[summary_df['GP'] == gp].copy()
    for _, row in gp_data.iterrows():
        color = TEAM_COLORS.get(row['Team'], '#888888')
        x = i + np.random.uniform(-0.15, 0.15)  # ジッターで重なりを軽減
        ax0.scatter(x, row['FuelCorrectedMedianPace'],
                    color=color, marker=marker_styles[i], s=60, alpha=0.85, zorder=3)

ax0.set_xticks(range(len(gp_list)))
ax0.set_xticklabels([GP_LABELS.get(g, g) for g in gp_list], color=TEXT, fontsize=9)
ax0.set_ylabel('燃料補正後ペース中央値（秒）', color=TEXT, fontsize=10)
ax0.set_title('ドライバー別 燃料補正後ペース（GP別）\n※補正後: LapTime + LapNumber×0.06秒（概算）',
              color=TEXT, fontsize=11, pad=8)

# チームカラー凡例（重複排除）
teams_in_data = summary_df['Team'].unique()
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=TEAM_COLORS.get(t, '#888888'), markersize=7, label=t)
    for t in TEAM_COLORS if t in teams_in_data
]
ax0.legend(handles=legend_handles, loc='upper right',
           facecolor=BG, edgecolor=GRID, labelcolor=TEXT, fontsize=7, ncol=2)

# ---- [1] チームランキング変動（バーグラフ） ----
ax1 = axes[1]
team_avg2 = team_df.groupby('Team')[['RawRank', 'FuelRank']].mean().reset_index().sort_values('FuelRank')
bar_w = 0.35
x = np.arange(len(team_avg2))

ax1.bar(x - bar_w/2, team_avg2['RawRank'],  bar_w, label='補正前ランク', color='#6688cc', alpha=0.8)
ax1.bar(x + bar_w/2, team_avg2['FuelRank'], bar_w, label='補正後ランク', color=RED, alpha=0.8)

ax1.set_xticks(x)
ax1.set_xticklabels(team_avg2['Team'], rotation=40, ha='right', color=TEXT, fontsize=8)
ax1.set_ylabel('平均ランキング（小=速い）', color=TEXT, fontsize=10)
ax1.set_title('チーム別 燃料補正前後のランキング（全GP平均）\n※補正前=生タイム、補正後=燃料補正済み（概算）',
              color=TEXT, fontsize=11, pad=8)
ax1.invert_yaxis()  # 1位が上
ax1.legend(facecolor=BG, edgecolor=GRID, labelcolor=TEXT, fontsize=9)

# ---- [2] 補正効果量の分布（ヒストグラム） ----
ax2 = axes[2]
for i, gp in enumerate(gp_list):
    gp_data = summary_df[summary_df['GP'] == gp]['PaceDiff']
    ax2.hist(gp_data, bins=8, alpha=0.6, color=gp_colors[i],
             label=GP_LABELS.get(gp, gp), edgecolor=GRID)

ax2.axvline(0, color=TEXT, linestyle='--', alpha=0.5, linewidth=1)
ax2.set_xlabel('補正後 − 補正前 ペース（秒）', color=TEXT, fontsize=10)
ax2.set_ylabel('ドライバー数', color=TEXT, fontsize=10)
ax2.set_title('燃料補正効果量の分布\n（正の値=補正後に大きく見える＝序盤ラップを多く使用）',
              color=TEXT, fontsize=11, pad=8)
ax2.legend(facecolor=BG, edgecolor=GRID, labelcolor=TEXT, fontsize=9)

# ---- [3] 補正後ペース vs レース最終順位 ----
ax3 = axes[3]
valid = summary_df.dropna(subset=['RacePosition', 'FuelCorrectedMedianPace'])

for i, gp in enumerate(gp_list):
    gp_data = valid[valid['GP'] == gp]
    ax3.scatter(gp_data['FuelCorrectedMedianPace'], gp_data['RacePosition'],
                color=gp_colors[i], s=60, alpha=0.85, zorder=3,
                label=GP_LABELS.get(gp, gp))
    for _, row in gp_data.iterrows():
        ax3.annotate(row['Driver'],
                     (row['FuelCorrectedMedianPace'], row['RacePosition']),
                     textcoords='offset points', xytext=(4, 0),
                     color=DIM, fontsize=6)

# 全GP合算の相関表示
if len(valid) > 2:
    rho_f, p_f = stats.spearmanr(valid['FuelCorrectedMedianPace'], valid['RacePosition'])
    ax3.text(0.05, 0.92, f'Spearman ρ = {rho_f:.3f} (p={p_f:.3f})',
             transform=ax3.transAxes, color=TEXT, fontsize=9,
             bbox=dict(facecolor=BG, alpha=0.7, edgecolor='none'))

ax3.set_xlabel('燃料補正後ペース中央値（秒）', color=TEXT, fontsize=10)
ax3.set_ylabel('レース最終順位', color=TEXT, fontsize=10)
ax3.invert_yaxis()  # 1位が上
ax3.set_title('燃料補正後ペース vs レース最終順位\n（概算補正・サーキット差含む）',
              color=TEXT, fontsize=11, pad=8)
ax3.legend(facecolor=BG, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# タイトル
fig.suptitle('F1 2026 R01-R03 燃料補正後レースペース分析\n'
             '（燃料補正値は概算: +0.06秒/ラップ。チームごとの燃料搭載量差は未考慮）',
             color=TEXT, fontsize=16, y=0.98, fontweight='bold')

# 保存
output_png = os.path.join(OUTPUT_DIR, 'fuel_pace_comparison.png')
plt.savefig(output_png, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'保存: {output_png}')

## 7. CSV出力

カラム: `GP, Driver, Team, RawMedianPace, FuelCorrectedMedianPace, CleanLaps, RacePosition`

In [ ]:
# 出力CSV（仕様書通りのカラム順）
output_cols = ['GP', 'Driver', 'Team', 'RawMedianPace', 'FuelCorrectedMedianPace', 'CleanLaps', 'RacePosition']
output_df = summary_df[output_cols].copy()

output_csv = os.path.join(OUTPUT_DIR, 'fuel_corrected_pace.csv')
output_df.to_csv(output_csv, index=False, encoding='utf-8')

print(f'保存: {output_csv}')
print(f'行数: {len(output_df)}')
output_df.head(10)

## まとめ

### 燃料補正の効果
- 補正量: ラップ番号 × 0.06秒（中央値≈1.7秒の加算）
- ランキング変動は最大±1〜2位程度
- 補正後でも相対順位は大きく変わらない（モデルが均一仮定のため）

### 注意点・制約
- **概算補正**: 0.06秒/ラップは中央値仮定。実際はチームごとに異なる
- **均一仮定**: 全ドライバーに同一の燃料消費量を適用
- **サーキット差非除去**: GP間の絶対値比較は不適切
- **相関係数の解釈**: ロングランのみのデータであり、予選・SC周回等は含まない

### 活用方法
- `fuel_corrected_pace.csv` をF1 Fantasy戦略やYouTube分析の入力として使用
- チームランキング変動の大きい場合は燃料戦略の差異を示唆する可能性
- 補正後ペースとレース順位の乖離 → 決勝戦略/SCの影響を示唆